# Exercise: N-BEATS Architecture Search

Compare different N-BEATS configurations on subscriber data. Vary input_chunk_length and output_chunk_length, then pick the best by validation MAE.

In [ ]:
import os
import warnings
os.environ["HF_HOME"] = os.path.expanduser("~/.cache/huggingface")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


In [ ]:
import matplotlib as mpl

UB = {"Brand Blue": "#175CFF", "Medium Blue": "#7BA2FF"}
UN = {"Black": "#0A0B0F", "Dark Gray": "#5A5C62"}
mpl.rcParams["lines.linewidth"] = 3
mpl.rcParams["axes.linewidth"] = 2


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from darts import TimeSeries
from darts.models import NBEATSModel
from darts.metrics import mae


In [ ]:
DATA_PATH = "../data/subscribers.csv"
HORIZON = 12


In [ ]:
def load_subscribers(path):
    """
    Load subscriber CSV and return train/val split.
    """
    df = pd.read_csv(path, parse_dates=..., index_col=...).asfreq("MS")
    ts = TimeSeries.from_series(df["subscribers"])
    return ts.split_after(pd.Timestamp("2023-12-01"))

train, val = load_subscribers(DATA_PATH)
print(f"Train: {len(train)} months, Val: {len(val)} months")


In [ ]:
def try_nbeats_config(train, val, input_length, output_length, epochs=30):
    """
    Fit N-BEATS with given chunk lengths and return validation MAE.
    """
    model = NBEATSModel(
        input_chunk_length=...,
        output_chunk_length=...,
        n_epochs=epochs,
        random_state=42,
        pl_trainer_kwargs={"enable_progress_bar": False}
    )
    model.fit(train)
    pred = model.predict(len(val))
    return mae(val, pred)

configs = [(12, 6), (24, 12), (36, 12)]
results = []
for inp, out in configs:
    err = try_nbeats_config(train, val, inp, out)
    results.append({"input": inp, "output": out, "mae": err})
    print(f"input={inp} output={out} => MAE={err:,.0f}")


In [ ]:
def plot_config_comparison(results):
    """
    Bar chart of MAE by configuration.
    """
    labels = [f"{r['input']}/{r['output']}" for r in results]
    values = [r["mae"] for r in results]
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(labels, values, color=UB["Brand Blue"])
    ax.set_title("N-BEATS Validation MAE by Configuration", fontsize=18, fontweight="bold")
    ax.set_ylabel("MAE (subscribers)", fontsize=16)
    ax.tick_params(axis="both", labelsize=14)
    plt.tight_layout()
    plt.show()

plot_config_comparison(results)
